# Task 8: Resume Screening Using NLP

In this notebook, we explore the datasets and test the semantic matching model (`SentenceTransformers`) used to screen resumes against job descriptions.

In [ ]:
import sys
import os
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from preprocess import clean_text
from ner import extract_entities
from matcher import ResumeMatcher

## 1. Load Data
We load the mock data generated in the `data/` directory.

In [ ]:
data_dir = os.path.join('..', 'data')
resumes_df = pd.read_csv(os.path.join(data_dir, 'resumes.csv'))
jobs_df = pd.read_csv(os.path.join(data_dir, 'jobs.csv'))

print("Resumes shape:", resumes_df.shape)
print("Jobs shape:", jobs_df.shape)

resumes_df.head()

## 2. Text Preprocessing and NER Extraction
Clean the text and extract technical skills using our custom rule-based and spaCy-based NER pipeline.

In [ ]:
resumes_df['clean_text'] = resumes_df['text'].apply(clean_text)
jobs_df['clean_description'] = jobs_df['description'].apply(clean_text)

# Extract skills as a list
resumes_df['extracted_skills'] = resumes_df['text'].apply(lambda t: extract_entities(t)['skills'])

resumes_df[['id', 'extracted_skills']]

## 3. Resume Matching via Sentence Transformers
We use `all-MiniLM-L6-v2` to compute the cosine similarity between the job description and candidate resumes.

In [ ]:
matcher = ResumeMatcher()

target_job = jobs_df.iloc[0]
print(f"Target Job: {target_job['title']}\n")

ranked_resumes = matcher.rank_resumes(
    job_desc=target_job['clean_description'],
    resumes=resumes_df['clean_text'].tolist(),
    resume_ids=resumes_df['id'].tolist()
)

pd.DataFrame(ranked_resumes)